# Building a semantic search engine

These abstractions are designed to support the retrieval of data from vector databases and other sources and intergrate them into LLM workflows. They are important for applications that fetch data to be reasoned over as part of the models interface such as in the case of Retreval-augmented generation.

This aims to build a search engine over a PDF document. 

*** Concepts ***:
1. Documents and document loaders
2. Text splitters
3. Embeddings
4. Vector stores and retrivers


# LangSmith

Helps inspecting whats going on within the chain/agent

In [7]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

## Documents and Document Loaders

LangChain implementas a Document abstraction, which is intended to represent a unit of text and associated metadata. it has three attributes:

1. page_content: a string representing the content
2. metadata: a dict containing arbitart metadata
3. id: an optional string identifier for the document

The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

In [8]:
from langchain_core.documents import Document

# Creating sample documents
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

### Loading the documents

Loading a PDF into a sequence of document objects. Using a sample document from the LangChain repo (A 10-k filing for Nike in 2023)

In [9]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "nike_10k_2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

106


In [10]:
print(f"{docs[1].page_content[:500]}\n")
print(docs[1].metadata)

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K 
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE FISCAL YEAR ENDED MAY 31, 2024  
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE TRANSITION PERIOD FROM                         TO                         .
Commission File No. 1-10635 
NIKE, Inc. 
(Exact name of Registrant as specified in its charter)
Oreg

{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-07-26T18:23:34+08:00', 'moddate': '2024-07-31T14:41:17+08:00', 'trapped': '/False', 'source': 'nike_10k_2023.pdf', 'total_pages': 106, 'page': 1, 'page_label': '2'}


## Splitting

The goal for this system is to filter through the documents information and find the relevant sections required to answer the query. Splitting our pdf will help ensure that the meanings of relevant portions of the document are not "washed out" by surrounding text.

For this we will use a simple text splitter that will split the document into 1000 character sections, with a 200 character overlay on each side (this is to mitigate the possibllity of seperating a statement from te important context related to it)

We use the RecursiveCharacterTextSplitter, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setting add_start_index to True so that the character index where each split document starts within intal document is preserved as metadata attribute "start_index"
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap = 200, add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

len(all_splits)

506

## Embeddings

Vector search is a common way to store and search over unstructured data.
The idea is to store numeric vectors that are associated with the text, then when given a query we can embed it as a vector of the same dimension and use vector similarity metrics (such as coosine similarity) to identifty related text.

In [16]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [17]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 3072

[0.0002851157041732222, -0.005536428187042475, -0.0056525482796132565, 0.018479643389582634, 0.01771656982600689, -0.03802095353603363, -0.0511259101331234, 0.0460166372358799, -0.02506529726088047, 0.08201378583908081]


## Vector stores

LangChain VectorStores objects contain methods for adding text and Document objects into the store and querying them using various similarity metrics. They're often initialised using embedding models which determine how the text data is translated into numeric vectors. 

In [18]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [19]:
ids = vector_store.add_documents(documents=all_splits)

## Usage

Embeddings typically represent text as a "dense" vector such as texts with similar meanings are geometrically close. T
his lets us retrieve relavent information just by passing in a question, without knowledge of any specific key-terms used in the document

In [20]:
# Returning documents based on similarity to string query

results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of 
which are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one 
located in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is 
located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of 
which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United 
States are located in Laakdal, Belgium; Taicang, China; Tomisato, Japan and Icheon, Korea, all of which we own.' metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 

In [23]:
#Async query

results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='NIKE, INC.
One Bowerman Drive
Beaverton, OR 97005-6453
www.nike.com
 ANNUAL REPORT AND NOTICE OF ANNUAL MEETING' metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.4 (Windows)', 'creationdate': '2024-07-26T18:23:34+08:00', 'moddate': '2024-07-31T14:41:17+08:00', 'trapped': '/False', 'source': 'nike_10k_2023.pdf', 'total_pages': 106, 'page': 105, 'page_label': '106', 'start_index': 0}


In [24]:
#Returning sources
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.696431591051332

page_content='FISCAL 2024 NIKE BRAND REVENUE HIGHLIGHTS
The following tables present NIKE Brand revenues disaggregated by reportable operating segment, distribution channel and 
major product line:
FISCAL 2024 COMPARED TO FISCAL 2023
• NIKE, Inc. Revenues for fiscal 2024 were $51.4 billion compared to $51.2 billion for fiscal 2023. On a currency-neutral basis, 
NIKE, Inc. Revenues increased 1%, as higher revenues in Greater China and Asia Pacific & Latin America ("APLA"), which 
each increased NIKE, Inc. Revenues by 1 percentage point, were partially offset by lower revenues in Converse, which 
reduced NIKE, Inc. Revenues by approximately 1 percentage point. 
• NIKE Brand revenues, which represented over 90% of NIKE, Inc. Revenues, increased 1% on both a reported and currency-
neutral basis. The increase, on a currency-neutral basis, was primarily due to higher revenues in the Jordan Brand and 
Men's.' metadata={'producer': 'Adobe PDF Library 17.0', 'creator':

In [ ]:
#Returning documents based on similarity to embedded query
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])